In [0]:
%sql
SELECT * FROM bronze.trades ORDER BY trade_id

In [0]:
%sql
DESCRIBE TABLE bronze.trades

In [0]:
%sql
SELECT COUNT(*) AS failing_rows
FROM bronze.trades
WHERE trade_id IS NULL
   OR TRIM(trade_id) = ''
   OR LOWER(TRIM(trade_id)) = 'null'

In [0]:
%sql
select count(*)as duplicate_id,trade_id from bronze.trades group by trade_id having count(*)>1

In [0]:
%sql
SELECT COUNT(*) AS failing_rows
FROM bronze.trades
WHERE trade_id IS NOT NULL
  AND NOT trade_id RLIKE '^T[0-9]+$'

In [0]:
%sql
SELECT COUNT(*) AS failing_ids
FROM (
  SELECT trade_id
  FROM bronze.trades
  GROUP BY trade_id
  HAVING COUNT(*) > 1
)

In [0]:
TABLE = "bronze.trades"

# (column, rule_type, parameter)
RULES = [
    ("trade_id", "not_blank", None),
    ("trade_id", "regex", r"^T[0-9]+$"),
    ("trade_id", "unique", None),
    ("portfolio_id", "not_blank", None),
    ("portfolio_id", "no_spaces", None),
    ("portfolio_id", "regex", r"^P[0-9]+$"),
    ("security_id", "not_blank", None),
    ("security_id", "regex", r"^SEC[0-9]+$"),
    ("trade_date", "valid_date", None),
    ("settle_date", "valid_date", None),
    ("settle_date", "not_before", "trade_date"),
    ("trade_type", "in_list", ["BUY", "SELL"]),
    ("quantity", "positive_number", None),
    ("price", "positive_number", None),
    ("currency", "in_list", ["USD", "GBP", "EUR"]),
    ("updated_at", "valid_timestamp", None),
]

In [0]:
from datetime import datetime

CONDITIONS = {  # each returns the "bad row" condition
    "not_blank": lambda c, p: f"{c} IS NULL OR TRIM({c}) = '' OR LOWER(TRIM({c})) = 'null'",
    "no_spaces": lambda c, p: f"{c} IS NOT NULL AND {c} != TRIM({c})",
    "regex": lambda c, p: f"{c} IS NOT NULL AND NOT {c} RLIKE '{p}'",
    "in_list": lambda c, p: f"{c} IS NOT NULL AND {c} NOT IN ({', '.join(repr(v) for v in p)})",
    "valid_date": lambda c, p: f"{c} IS NOT NULL AND TRY_CAST({c} AS DATE) IS NULL",
    "valid_timestamp": lambda c, p: f"{c} IS NOT NULL AND TRY_CAST({c} AS TIMESTAMP) IS NULL",
    "positive_number": lambda c, p: f"{c} IS NOT NULL AND (TRY_CAST({c} AS DECIMAL(18,4)) IS NULL OR TRY_CAST({c} AS DECIMAL(18,4)) <= 0)",
    "not_before": lambda c, p: f"{c} IS NOT NULL AND TRY_CAST({c} AS DATE) < TRY_CAST({p} AS DATE)",
}

def failing_sql(table, col, rule, param):
    if rule == "unique":
        return f"SELECT COUNT(*) AS n FROM (SELECT {col} FROM {table} GROUP BY {col} HAVING COUNT(*) > 1)"
    return f"SELECT COUNT(*) AS n FROM {table} WHERE {CONDITIONS[rule](col, param)}"

run_ts = datetime.now()
results = []
for col, rule, param in RULES:
    n = spark.sql(failing_sql(TABLE, col, rule, param)).collect()[0]["n"]
    results.append((TABLE, col, rule, str(param), n, "PASS" if n == 0 else "FAIL", run_ts))

df = spark.createDataFrame(
    results,
    ["table_name", "column_name", "rule", "parameter", "failing_rows", "status", "run_ts"],
)
display(df)

In [0]:
from datetime import datetime

TABLE = "bronze.trades"

RULES = [
    # trade_id
    ("trade_id not blank",   "trade_id IS NULL OR TRIM(trade_id) IN ('', 'null')"),
    ("trade_id format",      "trade_id IS NOT NULL AND trade_id NOT RLIKE '^T[0-9]+$'"),
    ("trade_id unique",      "trade_id IN (SELECT trade_id FROM bronze.trades GROUP BY trade_id HAVING COUNT(*) > 1)"),
    # portfolio_id
    ("portfolio_id not blank", "portfolio_id IS NULL OR TRIM(portfolio_id) IN ('', 'null')"),
    ("portfolio_id no spaces", "portfolio_id IS NOT NULL AND portfolio_id != TRIM(portfolio_id)"),
    ("portfolio_id format",    "portfolio_id IS NOT NULL AND portfolio_id NOT RLIKE '^P[0-9]+$'"),
    # security_id
    ("security_id not blank", "security_id IS NULL OR TRIM(security_id) IN ('', 'null')"),
    ("security_id format",    "security_id IS NOT NULL AND security_id NOT RLIKE '^SEC[0-9]+$'"),
    # dates
    ("trade_date not blank",   "trade_date IS NULL OR TRIM(trade_date) IN ('', 'null')"),
    ("trade_date valid date",  "trade_date IS NOT NULL AND TRY_CAST(trade_date AS DATE) IS NULL"),
    ("trade_date not future",  "TRY_CAST(trade_date AS DATE) > CURRENT_DATE()"),
    ("settle_date not blank",  "settle_date IS NULL OR TRIM(settle_date) IN ('', 'null')"),
    ("settle_date valid date", "settle_date IS NOT NULL AND TRY_CAST(settle_date AS DATE) IS NULL"),
    ("settle_date not before trade_date", "TRY_CAST(settle_date AS DATE) < TRY_CAST(trade_date AS DATE)"),
    # trade_type
    ("trade_type not blank",  "trade_type IS NULL OR TRIM(trade_type) IN ('', 'null')"),
    ("trade_type BUY or SELL", "trade_type IS NOT NULL AND trade_type NOT IN ('BUY', 'SELL')"),
    # numbers
    ("quantity not blank",    "quantity IS NULL OR TRIM(quantity) IN ('', 'null')"),
    ("quantity positive",     "quantity IS NOT NULL AND (TRY_CAST(quantity AS DECIMAL(18,4)) IS NULL OR TRY_CAST(quantity AS DECIMAL(18,4)) <= 0)"),
    ("price not blank",       "price IS NULL OR TRIM(price) IN ('', 'null')"),
    ("price positive",        "price IS NOT NULL AND (TRY_CAST(price AS DECIMAL(18,4)) IS NULL OR TRY_CAST(price AS DECIMAL(18,4)) <= 0)"),
    # currency
    ("currency not blank",    "currency IS NULL OR TRIM(currency) IN ('', 'null')"),
    ("currency format",       "currency IS NOT NULL AND currency NOT RLIKE '^[A-Z]{3}$'"),
    ("currency allowed",      "currency IS NOT NULL AND currency NOT IN ('USD', 'GBP', 'EUR')"),
    # updated_at
    ("updated_at not blank",  "updated_at IS NULL OR TRIM(updated_at) IN ('', 'null')"),
    ("updated_at valid",      "updated_at IS NOT NULL AND TRY_CAST(updated_at AS TIMESTAMP) IS NULL"),
    # audit columns
    ("_rescued_data is null", "_rescued_data IS NOT NULL"),
    ("_source_file not null", "_source_file IS NULL"),
    ("_ingested_at not null", "_ingested_at IS NULL"),
]

run_ts = datetime.now()
results = []
for name, bad_rows in RULES:
    n = spark.sql(f"SELECT COUNT(*) FROM {TABLE} WHERE {bad_rows}").collect()[0][0]
    results.append((TABLE, name, n, "PASS" if n == 0 else "FAIL", run_ts))

df = spark.createDataFrame(results, ["table_name", "rule", "failing_rows", "status", "run_ts"])
display(df)

In [0]:
t = spark.table(TABLE)
expected_cols = ["trade_id", "portfolio_id", "security_id", "trade_date", "settle_date",
                 "trade_type", "quantity", "price", "currency", "updated_at",
                 "_rescued_data", "_source_file", "_ingested_at"]
source_rows = spark.read.option("header", True).csv("/Volumes/workspace/bronze/landing/trades.csv").count()

print("row count > 0:      ", "PASS" if t.count() > 0 else "FAIL")
print("matches source file:", "PASS" if t.count() == source_rows else "FAIL", t.count(), source_rows)
print("column names match: ", "PASS" if t.columns == expected_cols else "FAIL")

In [0]:
spark.sql("CREATE SCHEMA IF NOT EXISTS dq")
df.write.mode("append").saveAsTable("dq.results")